In [ ]:
!pip install -q transformers accelerate sentencepiece torch

In [ ]:
from transformers import pipeline
import torch

print("GPU Available:", torch.cuda.is_available())

generator = pipeline(
    "text-generation",
    model="microsoft/Phi-3-mini-4k-instruct",
    device_map="auto"
)

print("Model Loaded Successfully")

In [ ]:
def generate_question():

    prompt = """
You are a technical interviewer.

Generate ONE technical interview question.

Topics can include:
- Python
- Java
- Machine Learning
- DBMS
- Data Structures
- Operating Systems
- Computer Networks
- Artificial Intelligence

Only return the question.
"""

    result = generator(
        prompt,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.9
    )

    return result[0]["generated_text"].replace(prompt, "").strip()

In [ ]:
def evaluate_answer(question, answer):

    prompt = f"""
You are a senior technical interviewer.

Question:
{question}

Candidate Answer:
{answer}

Evaluate the answer.

Provide output in this format:

Score: X/10

Strengths:
- point 1
- point 2

Weaknesses:
- point 1
- point 2

Correct Answer:
Give an ideal interview answer.
"""

    result = generator(
        prompt,
        max_new_tokens=300,
        do_sample=False
    )

    return result[0]["generated_text"].replace(prompt, "")

In [ ]:
all_questions = []
all_answers = []
all_feedbacks = []

for i in range(5):

    print("\n" + "="*70)

    question = generate_question()

    print(f"\nQuestion {i+1}")
    print(question)

    answer = input("\nYour Answer: ")

    feedback = evaluate_answer(
        question,
        answer
    )

    print("\nFeedback")
    print(feedback)

    all_questions.append(question)
    all_answers.append(answer)
    all_feedbacks.append(feedback)

In [ ]:
summary_prompt = f"""
You are an interview coach.

Questions:
{all_questions}

Answers:
{all_answers}

Generate:

Overall Score out of 100

Strong Areas

Weak Areas

Topics To Improve

Final Verdict
"""

report = generator(
    summary_prompt,
    max_new_tokens=400,
    do_sample=False
)

print("\nFINAL REPORT")
print("="*70)

print(report[0]["generated_text"])

In [ ]:
with open("Interview_Report.txt", "w") as f:

    f.write("AI Interview Coach Report\n")
    f.write("="*50 + "\n\n")

    for i in range(5):

        f.write(f"\nQuestion {i+1}\n")
        f.write(all_questions[i] + "\n\n")

        f.write("Answer:\n")
        f.write(all_answers[i] + "\n\n")

        f.write("Feedback:\n")
        f.write(all_feedbacks[i] + "\n\n")

    f.write("\nFINAL REPORT\n")
    f.write(report[0]["generated_text"])

print("Saved Successfully")

In [ ]:
from google.colab import files

files.download("Interview_Report.txt")

In [ ]:
from google.colab import output
output.disable_custom_widget_manager()